# 質問応答のデータセットを読み込んで中身を確認する (JSQuAD)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_qa/qa_dataset.ipynb)

NLP クックブック 19-01 ／ 質問応答

質問応答の学習や評価に使う日本語のデータセットを読み込み、質問・文書・答えがどんな形で入っているかを確かめたい。

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("sbintuitions/JSQuAD")     # JGLUE の読解データセット。Wikipedia の段落と、それに対する質問・答え
print(ds)
ex = ds["validation"][0]
for k, v in ex.items():
    print(f"{k}: {v}")

In [ ]:
# 答えは「文書中の文字位置」で与えられる。位置と文字列が合っているかを確かめる
ok = 0
for ex in ds["validation"].select(range(500)):
    for text, start in zip(ex["answers"]["text"], ex["answers"]["answer_start"]):
        ok += ex["context"][start:start + len(text)] == text
print("位置が合っている答え:", ok, "/", sum(len(ex["answers"]["text"]) for ex in ds["validation"].select(range(500))))

In [ ]:
import pandas as pd

df = ds["validation"].to_pandas()
df["context_len"] = df["context"].str.len()
df["question_len"] = df["question"].str.len()
df["answer_len"] = df["answers"].map(lambda a: len(a["text"][0]))
df["n_answers"] = df["answers"].map(lambda a: len(set(a["text"])))
print(df[["context_len", "question_len", "answer_len", "n_answers"]].describe().round(1).loc[["mean", "50%", "max"]])

# 質問の種類を、疑問詞でざっくり数える
import re
kinds = {"誰": r"誰|だれ", "いつ": r"いつ|何年|何月|何日|何世紀", "どこ": r"どこ|何処|何県|何市|どの国", "いくつ": r"いくつ|何人|何個|何回|いくら|どのくらい|何メートル", "何": r"何|なに|なん"}
def kind(q):
    return next((k for k, pat in kinds.items() if re.search(pat, q)), "その他")
print(df["question"].map(kind).value_counts())
print(df.sample(3, random_state=0)[["question"]].assign(answer=df["answers"].map(lambda a: a["text"][0])).to_string(index=False))

- SQuAD 形式: `context` (文書)、`question` (質問)、`answers` (答えの文字列と開始位置のリスト)。評価データには 1 つの質問に複数の正解が付いていることがある。
- JSQuAD の `context` は「記事タイトル [SEP] 段落」の形になっている。タイトルを使わないなら `[SEP]` より後ろを取り出す。
- JSQuAD の質問はすべて答えが文書中にある (SQuAD 1.1 相当)。答えのない質問を含む設定は「答えられない質問を判定する」のレシピを参照。
- 最近の `datasets` は、読み込みスクリプトを使う古い形式のデータセットを読めない。Parquet 形式で公開されているリポジトリを選ぶ (ここでは `sbintuitions/JSQuAD`)。